# Notebook 04: Congestion Risk Index
**Project:** Parking-Induced Congestion AI System

Since actual traffic speed data is unavailable, this notebook creates a **Parking Congestion Index** — a proxy risk score for each cluster.

### Scoring Components:
| Component | Weight |
|---|---|
| Violation count score | 35% |
| Peak-hour concentration | 20% |
| Violation type severity | 20% |
| Recurrence score | 15% |
| Junction proximity score | 10% |

Outputs:
- `congestion_risk_scores.csv`
- `enforcement_priority_zones.csv` (sorted by risk)

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported.')

Libraries imported.


## 2. Define Paths

In [2]:
CLUSTERED_CSV        = '../data/processed/parking_violations_with_clusters.csv'
HOTSPOT_CSV          = '../data/processed/hotspot_clusters.csv'
RISK_CSV             = '../data/processed/congestion_risk_scores.csv'
PRIORITY_CSV         = '../data/processed/enforcement_priority_zones.csv'

print('Paths ready.')

Paths ready.


## 3. Load Data

In [3]:
df_records  = pd.read_csv(CLUSTERED_CSV,  low_memory=False)
df_clusters = pd.read_csv(HOTSPOT_CSV,    low_memory=False)

if 'datetime' in df_records.columns:
    df_records['datetime'] = pd.to_datetime(df_records['datetime'], errors='coerce')

print(f'Clustered records : {df_records.shape}')
print(f'Hotspot clusters  : {df_clusters.shape}')
display(df_clusters.head())

Clustered records : (298450, 32)
Hotspot clusters  : (349, 11)


,cluster_id,violation_count,center_latitude,center_longitude,most_common_violation_type,most_common_police_station,most_common_location_or_junction,peak_hour,unique_days_count,active_months_count,severity_level
0,3,186628,12.978381,77.576195,"[""Wrong Parking""]",Upparpet,"Kamaraj Road, Sri Nagamma Devi Circle, Sivanch...",5.0,152,6,Critical
1,6,20204,12.936777,77.691050,"[""No Parking""]",Hal Old Airport,"New Horizon College Road, New Horizon College ...",22.0,152,6,Critical
2,0,11967,12.915845,77.627327,"[""Wrong Parking""]",Hsr Layout,"20Th Main Road, Block 7, Koramangala, Bengalur...",21.0,152,6,Critical
3,21,7006,13.039466,77.589575,"[""No Parking""]",Hebbala,"Bellary Road, Vinayaka Nagar, Hebbal, Bengalur...",5.0,151,6,Critical
4,28,6226,13.070997,77.588444,"[""Wrong Parking""]",Kodigehalli,"Sahakar Nagar Road, Fortuna Acacia, Byatarayan...",6.0,138,6,Critical


## 4. Score 1 – Violation Count Score (0-100)

In [4]:
def normalize_0_100(series):
    """Min-max normalize a series to 0-100."""
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series([50.0] * len(series), index=series.index)
    return ((series - mn) / (mx - mn) * 100).round(2)

df_risk = df_clusters.copy()
df_risk['violation_count_score'] = normalize_0_100(df_risk['violation_count'])
print('violation_count_score computed.')
display(df_risk[['cluster_id','violation_count','violation_count_score']].head())

violation_count_score computed.


,cluster_id,violation_count,violation_count_score
0,3,186628,100.00
1,6,20204,10.82
2,0,11967,6.41
3,21,7006,3.75
4,28,6226,3.33


## 5. Score 2 – Peak-Hour Score (0-100)

In [5]:
MORNING_PEAK = range(8,  12)   # 8-11
EVENING_PEAK = range(17, 22)   # 17-21
PEAK_HOURS   = list(MORNING_PEAK) + list(EVENING_PEAK)

valid_records = df_records[df_records['cluster_id'] != -1].copy()

if 'hour' in valid_records.columns and valid_records['hour'].notna().sum() > 0:
    valid_records['is_peak'] = valid_records['hour'].apply(
        lambda h: 1 if h in PEAK_HOURS else 0
    )
    peak_pct = valid_records.groupby('cluster_id')['is_peak'].mean().reset_index()
    peak_pct.columns = ['cluster_id', 'peak_hour_pct']
    peak_pct['peak_hour_score'] = (peak_pct['peak_hour_pct'] * 100).round(2)
    df_risk = df_risk.merge(peak_pct[['cluster_id','peak_hour_score']], on='cluster_id', how='left')
    df_risk['peak_hour_score'] = df_risk['peak_hour_score'].fillna(0)
    print('peak_hour_score computed.')
else:
    df_risk['peak_hour_score'] = 50.0
    print('WARNING: hour column missing. peak_hour_score set to 50.')

display(df_risk[['cluster_id','peak_hour_score']].head())

peak_hour_score computed.


,cluster_id,peak_hour_score
0,3,17.49
1,6,14.61
2,0,33.23
3,21,20.14
4,28,6.12


## 6. Score 3 – Violation Type Severity Score (0-100)

In [6]:
VIOLATION_SEVERITY_MAP = {
    'double parking':     100,
    'main road parking':  95,
    'no parking':         80,
    'footpath parking':   75,
    'obstruction':        70,
    'no standing':        65,
    'wrong parking':      55,
    'parking':            40,
    'parked':             35,
}

def get_violation_severity(text):
    """Return a severity score 0-100 based on violation type keywords."""
    if pd.isna(text):
        return 30.0
    text_lower = str(text).lower()
    best_score = 30.0
    for kw, score in VIOLATION_SEVERITY_MAP.items():
        if kw in text_lower:
            best_score = max(best_score, score)
    return float(best_score)

viol_col = 'most_common_violation_type' if 'most_common_violation_type' in df_risk.columns else None

if viol_col:
    df_risk['violation_type_severity_score'] = df_risk[viol_col].apply(get_violation_severity)
    print('violation_type_severity_score computed.')
else:
    df_risk['violation_type_severity_score'] = 40.0
    print('WARNING: most_common_violation_type missing. Default severity 40 used.')

display(df_risk[['cluster_id', viol_col if viol_col else 'cluster_id', 'violation_type_severity_score']].head())

violation_type_severity_score computed.


,cluster_id,most_common_violation_type,violation_type_severity_score
0,3,"[""Wrong Parking""]",55.0
1,6,"[""No Parking""]",80.0
2,0,"[""Wrong Parking""]",55.0
3,21,"[""No Parking""]",80.0
4,28,"[""Wrong Parking""]",55.0


## 7. Score 4 – Recurrence Score (0-100)

In [7]:
has_days   = 'unique_days_count'  in df_risk.columns
has_months = 'active_months_count' in df_risk.columns

if has_days and has_months:
    df_risk['unique_days_count']   = df_risk['unique_days_count'].fillna(1)
    df_risk['active_months_count'] = df_risk['active_months_count'].fillna(1)
    recurrence_raw = (
        0.6 * normalize_0_100(df_risk['unique_days_count']) +
        0.4 * normalize_0_100(df_risk['active_months_count'])
    )
    df_risk['recurrence_score'] = recurrence_raw.round(2)
    print('recurrence_score computed from unique_days_count + active_months_count.')
elif has_days:
    df_risk['unique_days_count'] = df_risk['unique_days_count'].fillna(1)
    df_risk['recurrence_score']  = normalize_0_100(df_risk['unique_days_count'])
    print('recurrence_score computed from unique_days_count only.')
else:
    df_risk['recurrence_score'] = 50.0
    print('WARNING: recurrence data missing. recurrence_score set to 50.')

display(df_risk[['cluster_id','recurrence_score']].head())

recurrence_score computed from unique_days_count + active_months_count.


,cluster_id,recurrence_score
0,3,100.00
1,6,100.00
2,0,100.00
3,21,99.60
4,28,94.44


## 8. Score 5 – Junction / High-Risk Area Score (0-100)

In [8]:
HIGH_RISK_KEYWORDS = [
    'junction', 'signal', 'circle', 'chowk', 'naka', 'cross', 'intersection',
    'bridge', 'metro', 'station', 'market', 'mall', 'school', 'hospital',
    'bus stop', 'railway', 'flyover', 'overpass', 'roundabout'
]

def compute_junction_score(text):
    """Score 0-100 based on how many high-risk keywords appear in location text."""
    if pd.isna(text):
        return 0.0
    text_lower = str(text).lower()
    hits = sum(1 for kw in HIGH_RISK_KEYWORDS if kw in text_lower)
    return min(hits * 25, 100)

loc_col = 'most_common_location_or_junction' if 'most_common_location_or_junction' in df_risk.columns else None

if loc_col and df_risk[loc_col].notna().sum() > 0:
    df_risk['junction_score'] = df_risk[loc_col].apply(compute_junction_score)
    print('junction_score computed from location_or_junction.')
else:
    df_risk['junction_score'] = 0.0
    print('WARNING: location_or_junction column not available. junction_score = 0.')

display(df_risk[['cluster_id','junction_score']].head())

junction_score computed from location_or_junction.


,cluster_id,junction_score
0,3,25.0
1,6,0.0
2,0,0.0
3,21,0.0
4,28,0.0


## 9. Final Parking Congestion Index (Weighted Sum)

In [9]:
W_COUNT    = 0.35
W_PEAK     = 0.20
W_SEVERITY = 0.20
W_RECUR    = 0.15
W_JUNC     = 0.10

df_risk['risk_score'] = (
    W_COUNT    * df_risk['violation_count_score'] +
    W_PEAK     * df_risk['peak_hour_score'] +
    W_SEVERITY * df_risk['violation_type_severity_score'] +
    W_RECUR    * df_risk['recurrence_score'] +
    W_JUNC     * df_risk['junction_score']
).round(2)

print(f'Risk score range: {df_risk["risk_score"].min():.1f}  to  {df_risk["risk_score"].max():.1f}')
print(f'Mean risk score : {df_risk["risk_score"].mean():.1f}')

Risk score range: 11.0  to  67.0
Mean risk score : 23.5


## 10. Risk Categories

In [10]:
def assign_risk_category(score):
    if score < 35:  return 'Low'
    if score < 60:  return 'Medium'
    if score < 80:  return 'High'
    return 'Critical'

df_risk['risk_category'] = df_risk['risk_score'].apply(assign_risk_category)
print('Risk category distribution:')
print(df_risk['risk_category'].value_counts())

Risk category distribution:
risk_category
Low       326
Medium     22
High        1
Name: count, dtype: int64


## 11. Recommended Enforcement Action

In [11]:
def recommend_action(row):
    """
    Generate a targeted enforcement recommendation based on risk score,
    peak-hour concentration, violation type severity, and junction proximity.
    """
    cat      = row.get('risk_category', 'Low')
    peak_s   = row.get('peak_hour_score', 0)
    junc_s   = row.get('junction_score', 0)
    sev_s    = row.get('violation_type_severity_score', 0)
    viol_txt = str(row.get('most_common_violation_type', '')).lower()
    loc_txt  = str(row.get('most_common_location_or_junction', '')).lower()

    if cat == 'Critical':
        if junc_s >= 50:
            return 'Deploy towing vehicle + temporary barricading near junction/metro/market'
        if 'double' in viol_txt:
            return 'Immediate towing of repeat double-parking violators + heavy fine enforcement'
        if peak_s >= 70:
            return 'Deploy enforcement team during morning and evening peak hours daily'
        return 'Intensive enforcement: towing + fines + daily patrol'

    if cat == 'High':
        if peak_s >= 60:
            return 'Deploy enforcement during peak hours (8-11 AM & 5-9 PM)'
        if junc_s >= 25:
            return 'Increase patrolling near junction/station; install temporary no-parking signs'
        return 'Add no-parking signage + regular tow enforcement'

    if cat == 'Medium':
        if 'market' in loc_txt or 'mall' in loc_txt:
            return 'Monitor during evening commercial peak; consider metered parking'
        return 'Periodic patrolling + no-parking awareness drive'

    # Low
    return 'Monitor and log; escalate if violations increase'

df_risk['recommended_enforcement_action'] = df_risk.apply(recommend_action, axis=1)
print('Recommendations generated.')
display(df_risk[['cluster_id','risk_score','risk_category','recommended_enforcement_action']].head(10))

Recommendations generated.


,cluster_id,risk_score,risk_category,recommended_enforcement_action
0,3,67.00,High,Increase patrolling near junction/station; ins...
1,6,37.71,Medium,Periodic patrolling + no-parking awareness drive
2,0,34.89,Low,Monitor and log; escalate if violations increase
3,21,36.28,Medium,Periodic patrolling + no-parking awareness drive
4,28,27.56,Low,Monitor and log; escalate if violations increase
5,10,35.29,Medium,Periodic patrolling + no-parking awareness drive
6,13,37.97,Medium,Periodic patrolling + no-parking awareness drive
7,4,39.56,Medium,Periodic patrolling + no-parking awareness drive
8,14,37.77,Medium,Periodic patrolling + no-parking awareness drive
9,35,29.82,Low,Monitor and log; escalate if violations increase


## 12. Save Outputs

In [12]:
# Full risk scores
df_risk.to_csv(RISK_CSV, index=False)
print(f'Congestion risk scores saved: {RISK_CSV}')

# Enforcement priority zones – sorted by risk descending
df_priority = df_risk.sort_values('risk_score', ascending=False).reset_index(drop=True)
df_priority.to_csv(PRIORITY_CSV, index=False)
print(f'Enforcement priority zones saved: {PRIORITY_CSV}')

Congestion risk scores saved: ../data/processed/congestion_risk_scores.csv
Enforcement priority zones saved: ../data/processed/enforcement_priority_zones.csv


## 13. Summary Printout

In [13]:
print('=' * 60)
print('CONGESTION RISK INDEX SUMMARY')
print('=' * 60)
print(f'  Total clusters scored       : {len(df_risk)}')
print(f'  Average risk score          : {df_risk["risk_score"].mean():.1f} / 100')
print(f'  Max risk score              : {df_risk["risk_score"].max():.1f}')
print()
print('Risk Category Counts:')
for cat in ['Critical','High','Medium','Low']:
    n = (df_risk['risk_category'] == cat).sum()
    pct = n / len(df_risk) * 100
    print(f'  {cat:12s}: {n:4d} ({pct:.1f}%)')
print()
print('Top 20 Priority Zones:')
disp_cols = [c for c in [
    'cluster_id','most_common_police_station','most_common_location_or_junction',
    'violation_count','risk_score','risk_category','recommended_enforcement_action'
] if c in df_priority.columns]
display(df_priority[disp_cols].head(20))

CONGESTION RISK INDEX SUMMARY
  Total clusters scored       : 349
  Average risk score          : 23.5 / 100
  Max risk score              : 67.0

Risk Category Counts:
  Critical    :    0 (0.0%)
  High        :    1 (0.3%)
  Medium      :   22 (6.3%)
  Low         :  326 (93.4%)

Top 20 Priority Zones:


,cluster_id,most_common_police_station,most_common_location_or_junction,violation_count,risk_score,risk_category,recommended_enforcement_action
0,3,Upparpet,"Kamaraj Road, Sri Nagamma Devi Circle, Sivanch...",186628,67.00,High,Increase patrolling near junction/station; ins...
1,4,K.R. Pura,"Mbt Road, Devasandra Junction, Kr Puram, Benga...",5596,39.56,Medium,Periodic patrolling + no-parking awareness drive
2,248,Byatarayanapura,"8Th Cross Road, Roshan Nagar, Deepanjali Nagar...",5,38.50,Medium,Periodic patrolling + no-parking awareness drive
3,25,Banaswadi,"Old Madras Road, Gopalan Signature Mall, Nagav...",956,38.12,Medium,Monitor during evening commercial peak; consid...
4,13,Mahadevapura,"Outer Ring Road, Venkatappa Colony, Mahadevapu...",5871,37.97,Medium,Periodic patrolling + no-parking awareness drive
5,40,Chikkajala,"Bengaluru Bellary Road, Sadahalli Gate Junctio...",959,37.87,Medium,Periodic patrolling + no-parking awareness drive
6,14,Chikkajala,"Unnamed Road, Begur Chikkanahalli, Bengaluru, ...",5145,37.77,Medium,Periodic patrolling + no-parking awareness drive
7,6,Hal Old Airport,"New Horizon College Road, New Horizon College ...",20204,37.71,Medium,Periodic patrolling + no-parking awareness drive
8,68,Peenya,"Jalahalli Cross Road, Jalahalli Cross Junction...",552,37.41,Medium,Periodic patrolling + no-parking awareness drive
9,108,Hennuru,"Thanisandra Main Road, Aswath Nagar Circle, Th...",162,36.97,Medium,Periodic patrolling + no-parking awareness drive
